### ЗАДАЧА: Реестр абонементов фитнес-клуба

Администратор фитнес-клуба получает строки с данными об абонементах.
Нужно собрать удобную модель, которая позволит:
- загрузить клиентов в единый реестр,
- посмотреть только активные абонементы,
- отфильтровать клиентов по тарифу,
- посчитать суммарное число оставшихся посещений,
- понять, как меняется реестр после активации и списания посещения.

В данных есть абонементы с разными статусами и остатком посещений,
поэтому важно корректно валидировать тариф, статус и изменение состояния объекта.


In [1]:
class Subscription:
    allowed_plans = {'standard', 'premium', 'family'}
    allowed_statuses = {'active', 'frozen', 'expired'}

    def __init__(self, sub_id, client_name, plan, visits_left, status):
        if plan not in self.allowed_plans:
            raise ValueError(f"Invalid plan: {plan}")
        if status not in self.allowed_statuses:
            raise ValueError(f"Invalid status: {status}")
        
        self.sub_id = sub_id
        self.client_name = client_name
        self.plan = plan
        self._visits_left = None
        self.visits_left = visits_left  

        self.status = status

    @property
    def visits_left(self):
        return self._visits_left

    @visits_left.setter
    def visits_left(self, value):
        value = int(value)
        if value < 0:
            raise ValueError('Visits must be >= 0')
        self._visits_left = value

    def use_visit(self):
        if self.status != 'active':
            raise ValueError("Cannot use visit; subscription is not active")
        if self.visits_left == 0:
            raise ValueError("No visits left to use")
        
        self._visits_left -= 1
        if self.visits_left == 0:
            self.status = 'expired'

    def freeze(self):
        if self.status == 'expired':
            raise ValueError("Cannot freeze an expired subscription")
        self.status = 'frozen'

    def activate(self):
        if self.visits_left == 0:
            raise ValueError("Cannot activate; no visits left")
        self.status = 'active'

    @classmethod
    def from_row(cls, row):
        parts = row.split('|')
        if len(parts) != 5:
            raise ValueError("Row must contain exactly 5 parts")
        sub_id, client_name, plan, visits_left, status = parts
        return cls(sub_id, client_name, plan, int(visits_left), status)

    def __repr__(self):
        return f"Subscription(sub_id='{self.sub_id}', client_name='{self.client_name}', status='{self.status}')"


class SubscriptionRegistry:
    def __init__(self):
        self.items = []

    def add(self, subscription):
        self.items.append(subscription)

    def load(self, rows):
        for row in rows:
            subscription = Subscription.from_row(row)
            self.add(subscription)

    def active_subscriptions(self):
        return [sub for sub in self.items if sub.status == 'active']

    def by_plan(self, plan):
        return [sub for sub in self.items if sub.plan == plan]

    def total_visits_left(self):
        return sum(sub.visits_left for sub in self.items)

    def status_summary(self):
        summary = {}
        for sub in self.items:
            if sub.status in summary:
                summary[sub.status] += 1
            else:
                summary[sub.status] = 1
        return summary

    def find(self, sub_id):
        for sub in self.items:
            if sub.sub_id == sub_id:
                return sub
        return None



rows = [
    'SB-100|Alice|standard|8|active',
    'SB-101|Bob|premium|12|frozen',
    'SB-102|Charlie|family|0|expired',
    'SB-103|Diana|standard|5|active'
]

registry = SubscriptionRegistry()
registry.load(rows)


print(registry.items)


print(registry.active_subscriptions())


print(registry.by_plan('standard'))


print(registry.total_visits_left())


print(registry.status_summary())


sub_101 = registry.find('SB-101')
if sub_101:
    sub_101.activate()
print(registry.status_summary())


sub_100 = registry.find('SB-100')
if sub_100:
    sub_100.use_visit()
print(sub_100)


[Subscription(sub_id='SB-100', client_name='Alice', status='active'), Subscription(sub_id='SB-101', client_name='Bob', status='frozen'), Subscription(sub_id='SB-102', client_name='Charlie', status='expired'), Subscription(sub_id='SB-103', client_name='Diana', status='active')]
[Subscription(sub_id='SB-100', client_name='Alice', status='active'), Subscription(sub_id='SB-103', client_name='Diana', status='active')]
[Subscription(sub_id='SB-100', client_name='Alice', status='active'), Subscription(sub_id='SB-103', client_name='Diana', status='active')]
25
{'active': 2, 'frozen': 1, 'expired': 1}
{'active': 3, 'expired': 1}
Subscription(sub_id='SB-100', client_name='Alice', status='active')
